# Settings

In [1]:
parameters_file_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/params.json"
convert_complex = True
apply_directional_filter = False     #!convert_complex must be True for this
directional_filter_direction = "up"  # up, down
apply_hp_svd = False

#Simulation settings
using_simulation_data = False
rf_matrix_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/concat/1760_concat_rf.npy"

#Real data settings
using_matlab_data = False       #! high RAM usage for big files -> convert mat file to npy for memory map usage and set using_simulation_data to False
matlab_rf_data_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/rf0001.mat"
interpolate_data = True

#Batch processing settings
batch_processing = True
batch_size = 2                 #ignored if batch_processing is False
frames_to_process = 1760
MAX_WORKERS = 10                 #max is 12 for the current computer (10 max recommended)

#Saving parameters
save_image = True
rf_data_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/concat/"
folder_name = "IQ"
processed_rf_matrix_name = "1760_concat_IQ_rf"


# Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import timeit
import os
import math
import concurrent.futures
from numba import njit
from scipy.signal import firwin, hilbert, butter, lfilter
from scipy.io import loadmat

# Import parameters

In [3]:
def load_parameters(json_file_path):
    with open(json_file_path, 'r') as f:
        params = json.load(f)
    return params


In [4]:
def add_extra_params(params):
    
    params["rf"]["upsampled_samples"] = params["rf"]["samples"] + (params["rf"]["interpolation_factor"] - 1) * (params["rf"]["samples"] - 1)
    params["rf"]["upsampled_sampling_rate"] = params["rf"]["sampling_rate"] * params["rf"]["interpolation_factor"] 

    return params

In [5]:
params = load_parameters(parameters_file_path)
params = add_extra_params(params)

pos = np.linspace(start=0, stop=params["array"]["length"], num=params["array"]["length"], dtype=int) #array position sampling
time = np.linspace(start=0, stop=1 / params["rf"]["sampling_rate"] * params["rf"]["samples"], num=params["rf"]["samples"]) #time sampling
C = params["rf"]["C"] #speed of sound

print(f"Max acquisition depth : {params["array"]["acq_depth"][1]} m") #v = d / t => d = v * t / 2 (round trip) with t = sample_step * samples

Max acquisition depth : 0.012 m


# Create workspace

In [6]:
if not os.path.exists(f"{rf_data_path}{folder_name}"):
    os.makedirs(f"{rf_data_path}{folder_name}")
    print(f"Folder {rf_data_path}{folder_name} doesn't exist. Setting up the folder.")

# Import RF matrix

In [7]:
def load_rf_data(rf_matrix_path, matlab_rf_data_path, desired_frames : list, using_simulation_data, using_matlab_data, nb_angles):
    debug_statement = ""

    if using_simulation_data:

        rf_matrix_summed = np.load(rf_matrix_path)
        if rf_matrix_summed.ndim == 3:
                rf_matrix_summed = rf_matrix_summed[:, :, :, np.newaxis]
        rf_matrix_summed = rf_matrix_summed[:,:,:,desired_frames[0]:desired_frames[1]]
        debug_statement += f"Imported RF matrix with shape:  {rf_matrix_summed.shape}"

    else:
        if using_matlab_data:
            rf_matrix_summed = loadmat(matlab_rf_data_path)
            rf_matrix_summed = list(rf_matrix_summed.values())[-1]
            if rf_matrix_summed.ndim == 3:
                rf_matrix_summed = rf_matrix_summed[:, :, :, np.newaxis]
            rf_matrix_summed = rf_matrix_summed[:,:,:,desired_frames[0]:desired_frames[1]]
            rf_matrix_summed = np.float32(rf_matrix_summed)
            debug_statement += f"Imported RF matrix with shape: {rf_matrix_summed.shape}"
        else:
            rf_matrix_summed = np.lib.format.open_memmap(rf_matrix_path, mode='r')
            rf_matrix_summed = rf_matrix_summed[:,:,:,desired_frames[0]:desired_frames[1]]
            # if rf_matrix_summed.ndim == 3:
            #         rf_matrix_summed = rf_matrix_summed[:, :, :, np.newaxis]
            debug_statement += f"Imported RF matrix with shape:  {rf_matrix_summed.shape}"
            

    return rf_matrix_summed, debug_statement

# Interpolation 

In [8]:
@njit()
def zero_stuffing(rf_matrices_summed, interpolation_factor, upsampled_samples):
    interpolation_factor = interpolation_factor - 1
    
    new_matrices_summed = np.zeros((upsampled_samples, rf_matrices_summed.shape[1], rf_matrices_summed.shape[2]), dtype=rf_matrices_summed.dtype)
    
    for i in range(rf_matrices_summed.shape[2]): #for each angle
        for j in range(rf_matrices_summed.shape[1]): #for each transducer
            for k in range(rf_matrices_summed.shape[0]): #for each sample
                idx_new = k + (interpolation_factor * k)
                new_matrices_summed[idx_new, j, i] = rf_matrices_summed[k, j, i]

    return new_matrices_summed

In [9]:
def get_filtered_matrices(zero_stuffed_rf_matrices_summed, sampling_frequency, interpolation_factor):
    filtered_rf_matrices_summed = np.zeros((zero_stuffed_rf_matrices_summed.shape[0], zero_stuffed_rf_matrices_summed.shape[1], zero_stuffed_rf_matrices_summed.shape[2]), dtype=zero_stuffed_rf_matrices_summed.dtype)
    
    n_taps = 129
    cutoff_freq = sampling_frequency / 2

    filtering_function = firwin(n_taps, cutoff_freq, fs=sampling_frequency * interpolation_factor, window='hamming') * interpolation_factor

    for i in range(0, zero_stuffed_rf_matrices_summed.shape[2]): #for each angle
        for j in range(0, zero_stuffed_rf_matrices_summed.shape[1]): #for each transducer
            filtered_rf_matrices_summed[:,j,i] = np.convolve(zero_stuffed_rf_matrices_summed[:,j,i], filtering_function, mode='same')

    return filtered_rf_matrices_summed

# IQ data

In [10]:
def get_complex(rf_data, frames):
    complex_rf_data = np.zeros_like(rf_data, dtype=np.complex64)

    complex_rf_data = hilbert(rf_data, axis=0)

    return complex_rf_data

# Filters

## HP

In [11]:
def filter_hp(bf, fsam, f, nrebound):

    #subtract continuous componen
    bf  = bf -  np.mean(bf, axis=2, keepdims=True)
    #butterworth high-pass filter
    b, a = butter(3, f*0.6, 'highpass', output='ba', fs=fsam)
    #apply filter along the time dimension
    bff = lfilter(b, a, bf, axis=2)
    #the first nelim images are eliminated because the filter border effect
    if nrebound > 0:
        bff[:,:,0:nrebound] = 0
        
    return bff

## SVD

In [12]:
def filter_svd(bf, nelim):
    if nelim == 0:
        return bf, None
    #get dimensions of data
    nz, nx, nt = bf.shape
    #reshape to 2D matrix (space-time)
    m = np.reshape(bf, (nz*nx, nt))
    #remove mean from each row
    m = m - np.mean(m, axis=1, keepdims=True)
    #perform SVD decomposition
    U, S, Vt = np.linalg.svd(m, full_matrices=False)
    #create diagonal matrix from singular values
    Sf = np.diag(S)
    #set smallest singular values to zero (eliminating noise)
    Sf[:nelim, :nelim] = 0
    #reconstruct filtered matrix 
    mf = U @ Sf @ Vt
    #reshape back to 3D
    bff = np.reshape(mf, (nz, nx, nt))
    #return filtered data and singular values
    return bff, S

## Directional filter

In [13]:
def filter_movement(rf_data, frames, direction = "up"):
    filtered_rf_data = np.zeros_like(rf_data, dtype=np.complex64)

    instantaneous_phase = np.angle(rf_data)

    for h in range(0, rf_data.shape[2]):    #angles
        for i in range(0, rf_data.shape[1]):    #tranducers
            for j in range(0, rf_data.shape[0]):    #samples depth
                for k in range(1 , frames):  #frames

                    phase_shift = instantaneous_phase[j,i,h,k] - instantaneous_phase[j,i,h,k-1]
                    if direction == "up":
                        filtered_rf_data[j,i,h,k] = 0 if phase_shift > 0 else rf_data[j,i,h,k]
                    elif direction == "down":
                        filtered_rf_data[j,i,h,k] = 0 if phase_shift < 0 else rf_data[j,i,h,k]

    
    return filtered_rf_data

# Run preprocessing pipeline

In [14]:
def process_batch_multiprocessing(
    batch_idx, start_frame, current_batch_size, total_frames, frame_iterations_total,
    rf_matrix_path_arg, matlab_rf_data_path_arg, using_simulation_data_arg, using_matlab_data_arg,
    interpolate_data_arg, params_arg, samples_arg, sampling_rate_arg, rf_data_path_arg, folder_name_arg, processed_rf_matrix_name_arg,
    rf_dtype_arg
    ):
    """
    Processes a single batch of RF data frames in a separate process.

    Returns:
        tuple: (batch_idx, True) if successful, (batch_idx, False) otherwise.
    """
    try:
        #Load rf data for the current batch

        rf_matrix, debug_statement = load_rf_data(
            rf_matrix_path_arg,
            matlab_rf_data_path_arg,
            [start_frame, start_frame + current_batch_size],
            using_simulation_data_arg,
            using_matlab_data_arg,
            len(params_arg["rf"]["angles"])
        )

        print(f"{'='*60}\nProcessing batch {batch_idx} [{batch_idx}/{frame_iterations_total}] (Process ID: {os.getpid()})"
            f"\n  • Processing from frame {start_frame} to frame {start_frame + current_batch_size}"
            f"\n  • Processing {current_batch_size} frame(s)"
            f"\n  • RF matrix: {debug_statement}"
            f"\n{'='*60}\n")

        actual_loaded_frames = rf_matrix.shape[3]

        if apply_hp_svd:
            for i in range(0, len(params_arg["rf"]["angles"])):
                rf_matrix[:, :, i, :] = filter_hp(rf_matrix[:, :, i, :], sampling_rate_arg , params_arg["rf"]["frequency"], 0)
                rf_matrix[:, :, i, :], _ = filter_svd(rf_matrix[:, :, i, :], 1)

        if convert_complex:

            rf_matrix = get_complex(rf_matrix, actual_loaded_frames)

            if apply_directional_filter:
                rf_matrix = filter_movement(rf_matrix, actual_loaded_frames, direction=directional_filter_direction)

        if actual_loaded_frames != current_batch_size:
             print(f"Warning: Loaded {actual_loaded_frames} frames for batch {batch_idx}, expected {current_batch_size}.")
             current_batch_size = actual_loaded_frames

        #Interpolate data
        if interpolate_data_arg:
            rf_matrix_tmp = np.zeros((params_arg["rf"]["upsampled_samples"], rf_matrix.shape[1], rf_matrix.shape[2], actual_loaded_frames), dtype=rf_dtype_arg)

            for j in range(actual_loaded_frames):
                temp_zero_stuffed = zero_stuffing(rf_matrix[:,:,:,j], params_arg["rf"]["interpolation_factor"], params_arg["rf"]["upsampled_samples"])
                rf_matrix_tmp[:,:,:,j] = get_filtered_matrices(temp_zero_stuffed, params_arg["rf"]["sampling_rate"], params_arg["rf"]["interpolation_factor"])

            rf_matrix = rf_matrix_tmp

            del temp_zero_stuffed
            del rf_matrix_tmp

        output_folder = os.path.join(rf_data_path_arg, folder_name_arg)

        memmap_path = os.path.join(output_folder, f"{processed_rf_matrix_name_arg}.npy")
        memmap_rf_matrix = np.lib.format.open_memmap(memmap_path, dtype=rf_dtype_arg, mode='r+', shape=(samples_arg, params_arg["array"]["length"], len(params_arg["rf"]["angles"]), total_frames))
        memmap_rf_matrix[:, :, :, start_frame : start_frame + actual_loaded_frames] = rf_matrix
        memmap_rf_matrix.flush() #making sure data is written to disk

        del memmap_rf_matrix    #force garbage collection
        del rf_matrix

        print("="*60 + "\n" + f"Finished processing batch {batch_idx} (Process ID: {os.getpid()})\n" + "="*60 + "\n")
        return (batch_idx, True) #return process status for easier debuging

    except Exception as e:
        print(f"!!! Error in process {os.getpid()} processing batch {batch_idx} (frames {start_frame} to {start_frame + current_batch_size}): {e}")
        import traceback
        traceback.print_exc()
        return (batch_idx, False)

## Run

In [15]:
def save_data(matrix, filename, destination=f"{rf_data_path}/{folder_name}/"):
    np.save(f"{destination}{filename}", matrix)
    print(f"Data of shape {matrix.shape} saved at: {destination}{filename}.npy")

In [16]:
batch_size = batch_size if batch_processing else 1

#load right parameters for batch processing 
if frames_to_process % batch_size == 0:
    frame_iterations = frames_to_process // batch_size
else:
    frame_iterations = frames_to_process // batch_size + 1

print(f"Frame iterations: {frame_iterations}")
if batch_processing:
    print(f"Batch processing with {batch_size} frames")

if convert_complex:
    rf_dtype = np.complex64
else:
    rf_dtype = np.float32

if interpolate_data:
    samples = params["rf"]["upsampled_samples"]
    sampling_rate = params["rf"]["upsampled_sampling_rate"]
else:
    samples = params["rf"]["samples"]
    sampling_rate = params["rf"]["sampling_rate"]

#create numpy matrix for batch processing
#rf_matrix_preprocessed = np.zeros((samples, params["array"]["length"], len(params["rf"]["angles"]), frames_to_process), dtype=rf_dtype)
rf_matrix_preprocessed = np.lib.format.open_memmap(f"{rf_data_path}{folder_name}/{processed_rf_matrix_name}.npy", dtype=rf_dtype, mode='w+', shape=(samples, params["array"]["length"], len(params["rf"]["angles"]), frames_to_process))
print(f"Created memmap file with shape: {rf_matrix_preprocessed.shape}")
#save_data(rf_matrix_preprocessed, f"{processed_rf_matrix_name}", destination=f"{rf_data_path}{folder_name}/")

Frame iterations: 880
Batch processing with 2 frames
Created memmap file with shape: (1535, 128, 16, 1760)


In [17]:
#run multiprocessing
start_time = timeit.default_timer()

print(f"\nStarting process pool with {MAX_WORKERS} workers.")
print(f"Total frames to process: {frames_to_process}")
print(f"Batch size: {batch_size}")
print(f"Total batches: {frame_iterations}")
futures = []

with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    last_batch_flag = False

    for i in range(0, frames_to_process, batch_size):
        if last_batch_flag:
            break #for safety
        current_batch_start_frame = i

        # last batch nb of frames if batch size isnt multiple of frames_to process - spot last batch
        if current_batch_start_frame + batch_size >= frames_to_process:
            current_batch_actual_size = frames_to_process - current_batch_start_frame
            last_batch_flag = True
            print(f"Last batch identified: Start Frame {current_batch_start_frame}, Size {current_batch_actual_size}")
        else:
            current_batch_actual_size = batch_size
            #set last batch if we have batch size multiple of frames to process
            last_batch_flag = (current_batch_start_frame + batch_size == frames_to_process)

        if current_batch_actual_size <= 0:
            print(f"Skipping batch starting at frame {current_batch_start_frame} due to zero or negative size.")
            continue

        batch_index = int(i / batch_size) + 1

        #submit task to process pool
        futures.append(executor.submit(
            process_batch_multiprocessing,
            #batch args
            batch_idx=batch_index,
            start_frame=current_batch_start_frame,
            current_batch_size=current_batch_actual_size,
            total_frames=frames_to_process,
            frame_iterations_total=frame_iterations,
            #other args
            rf_matrix_path_arg=rf_matrix_path,
            matlab_rf_data_path_arg=matlab_rf_data_path,
            using_simulation_data_arg=using_simulation_data,
            using_matlab_data_arg=using_matlab_data,
            interpolate_data_arg=interpolate_data,
            params_arg=params,
            samples_arg=samples,
            sampling_rate_arg=sampling_rate,
            rf_data_path_arg=rf_data_path,
            folder_name_arg=folder_name,
            processed_rf_matrix_name_arg =processed_rf_matrix_name,
            rf_dtype_arg=rf_dtype
        ))

    #evluate all processes
    print("\nWaiting for all batches to complete...\n")
    results = []
    num_success = 0
    num_failed = 0
    for future in concurrent.futures.as_completed(futures):
        try:
            batch_num, success = future.result() #get function output 
            if success:
                num_success += 1
            else:
                num_failed += 1
                print(f"Batch {batch_num} failed.")
            results.append(success)
        except Exception as exc:
            print(f'A batch generated an exception during execution/retrieval: {exc}')
            num_failed += 1
            results.append(False)

#end of multiprocessing
end_time = timeit.default_timer()

print("\n--- Processing Summary ---\n")
print(f"Total Batches: {frame_iterations}")
print(f"Successful Batches: {num_success}")
print(f"Failed Batches: {num_failed}")
if num_failed == 0:
    print("\nAll batches processed successfully.")
else:
    print(f"\n{num_failed} batch(es) failed. Check logs above.")
print(f"\nTotal Beamforming took: {end_time - start_time:.2f} seconds")


Starting process pool with 10 workers.
Total frames to process: 1760
Batch size: 2
Total batches: 880


Processing batch 1 [1/880] (Process ID: 69160)
  • Processing from frame 0 to frame 2
  • Processing 2 frame(s)
  • RF matrix: Imported RF matrix with shape:  (768, 128, 16, 2)
Processing batch 5 [5/880] (Process ID: 69168)
  • Processing from frame 8 to frame 10
  • Processing 2 frame(s)
  • RF matrix: Imported RF matrix with shape:  (768, 128, 16, 2)
Processing batch 2 [2/880] (Process ID: 69162)
  • Processing from frame 2 to frame 4
  • Processing 2 frame(s)
  • RF matrix: Imported RF matrix with shape:  (768, 128, 16, 2)
Processing batch 8 [8/880] (Process ID: 69173)
  • Processing from frame 14 to frame 16
  • Processing 2 frame(s)
  • RF matrix: Imported RF matrix with shape:  (768, 128, 16, 2)
Processing batch 9 [9/880] (Process ID: 69174)
  • Processing from frame 16 to frame 18
  • Processing 2 frame(s)
  • RF matrix: Imported RF matrix with shape:  (768, 128, 16, 2)
Processing batch 4 [4/880] (Process ID: 69167)
  • Processing from frame 6 to frame 8
  • Processing 2 frame(s